# Step 19: Pretrained Transformer features
The NoXi zips also contain features made by six big pretrained Transformers (text: XLM-RoBERTa; audio: w2v-BERT 2.0; video: CLIP, DINOv2, Swin, VideoMAE). We average them over the same 1-second windows as step 2.
*Same as `src/19_pretrained_features.py`; compared with the saved files. Slow: it reads the 14 GB zip.*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")          # work from the project folder

In [ ]:
import zipfile
import warnings
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")      # "mean of empty slice" warnings

EMBEDDINGS = {"xlm_roberta": "audio.xlm_roberta_embeddings", "w2vbert2": "audio.w2vbert2_embeddings",
              "clip": "clip", "dino": "dino", "swin": "swin", "videomae": "videomae"}

**Read a feature file** from the folder (like step 2) **or directly from the zip**.

In [ ]:
def read_stream(path):
    dim = int(ET.parse(path).getroot().find("info").get("dim"))
    data = np.fromfile(path + "~", dtype=np.float32)
    return data[: data.size // dim * dim].reshape(-1, dim)


def read_from_zip(z, name):
    dim = int(ET.fromstring(z.read(name)).find("info").get("dim"))
    data = np.frombuffer(z.read(name + "~"), dtype=np.float32)
    return data[: data.size // dim * dim].reshape(-1, dim)

Session 027 is in its own zip; all others are in train.zip.

In [ ]:
train_zip = zipfile.ZipFile("C:/Users/SHUBHAM KUMAR/Downloads/train.zip")
zip_027 = zipfile.ZipFile("C:/Users/SHUBHAM KUMAR/Downloads/027.zip")

**Main loop:** find the same windows as step 2, then average every embedding over those windows.

In [ ]:
result = {}
for key in EMBEDDINGS:
    result[key] = []

for session in sorted(os.listdir("data/noxi")):
    folder = "data/noxi/" + session
    z = zip_027 if session == "027" else train_zip
    for role in ["expert", "novice"]:
        # the windows kept in step 2
        voice = read_stream(folder + "/" + role + ".audio.egemapsv2.stream")
        face = read_stream(folder + "/" + role + ".openface2.stream")
        label = pd.read_csv(folder + "/" + role + ".engagement.annotation.csv", header=None)[0]
        label = pd.to_numeric(label, errors="coerce").to_numpy("float32")
        n = min(len(voice), len(face), len(label))
        keep = []
        for w in range(n // 25):
            if not np.isnan(np.nanmean(label[w * 25:(w + 1) * 25])):
                keep.append(w)

        # average each embedding over the same windows
        for key, stream in EMBEDDINGS.items():
            E = read_from_zip(z, session + "/" + role + "." + stream + ".stream")
            windows = []
            for w in keep:
                if w * 25 < len(E):
                    windows.append(np.nanmean(E[w * 25:(w + 1) * 25], axis=0))
                else:
                    windows.append(np.full(E.shape[1], np.nan, np.float32))    # embedding is shorter
            result[key].append(np.nan_to_num(np.stack(windows)).astype(np.float32))
    print(session, "done")

**Same as the saved files?** (True for all six)

In [ ]:
for key in EMBEDDINGS:
    new = np.concatenate(result[key])
    saved = np.load("data/features/pretrained_" + key + ".npy")
    print(key.ljust(12), new.shape, np.array_equal(new, saved))